In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

About Dataset

This synthetic dataset is modeled after an existing milling machine and consists of 10 000 data points from a stored as rows with 14 features in columns

1.UID: unique identifier ranging from 1 to 10000

2.product ID: consisting of a letter L, M, or H for low (50% of all products), medium (30%) and high (20%) as product quality variants and a variant-specific serial number

3.type: just the product type L, M or H from column 2

4.air temperature [K]: generated using a random walk process later normalized to a standard deviation of 2 K around 300 K

5.process temperature [K]: generated using a random walk process normalized to a standard deviation of 1 K, added to the air temperature plus 10 K.

6.rotational speed [rpm]: calculated from a power of 2860 W, overlaid with a normally distributed noise

7.torque [Nm]: torque values are normally distributed around 40 Nm with a SD = 10 Nm and no negative values.

8.tool wear [min]: The quality variants H/M/L add 5/3/2 minutes of tool wear to the used tool in the process.

9.a 'machine failure' label that indicates, whether the machine has failed in this particular datapoint for any of the following failure modes are true.



The machine failure consists of five independent failure modes

1.tool wear failure (TWF): the tool will be replaced of fail at a randomly selected tool wear time between 200 - 240 mins (120 times in our dataset). At this point in time, the tool is replaced 69 times, and fails 51 times (randomly assigned).

2.heat dissipation failure (HDF): heat dissipation causes a process failure, if the difference between air- and process temperature is below 8.6 K and the tools rotational speed is below 1380 rpm. This is the case for 115 data points.

3.power failure (PWF): the product of torque and rotational speed (in rad/s) equals the power required for the process. If this power is below 3500 W or above 9000 W, the process fails, which is the case 95 times in our dataset.

4.overstrain failure (OSF): if the product of tool wear and torque exceeds 11,000 minNm for the L product variant (12,000 M, 13,000 H), the process fails due to overstrain. This is true for 98 datapoints.

5.random failures (RNF): each process has a chance of 0,1 % to fail regardless of its process parameters. This is the case for only 5 datapoints, less than could be expected for 10,000 datapoints in our dataset.

If at least one of the above failure modes is true, the process fails and the 'machine failure' label is set to 1. It is therefore not transparent to the machine learning method, which of the failure modes has caused the process to fail.

# 1. Learn about basic information about the data set

* 1.1 Import relevant packages and set parameters for visualization

In [ ]:
import pandas as pd
import numpy as np

from IPython.core.display import display, HTML
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio  

import seaborn as sns
from importlib import reload
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", 500)
pd.set_option('display.expand_frame_repr', False)  
display(HTML("<style>div.output_scroll { height: 35em; }</style>"))

pio.renderers.default = 'iframe'
pio.templates['ck_template'] = go.layout.Template(
    layout_colorway=px.colors.sequential.Viridis,
    layout_autosize=False,
    layout_width=800,
    layout_height=600,
    layout_font=dict(family="Calibri Light"),
    layout_title_font=dict(family="Calibri"),
    layout_hoverlabel_font=dict(family="Calibri Light")
)

pio.templates.default = 'ck_template+gridon'

* 1.2 Load data set and print some basic information

In [ ]:
df = pd.read_csv('/kaggle/input/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv')
df.info(),df.head()

In [ ]:
for column in df.columns:
    try:
        df[column] = df[column].astype(float)
    except:
        pass
    
# Perform a data type conversion
print(df.info(), df.head())  
df_numeric = df.select_dtypes(include=[np.number])
print(df_numeric.describe(include='all').T)   # In addition to describe continuous data, we also describe discrete data

* 1.3 Draw relationship figures between attributes and factors

In [ ]:
plt.figure(figsize=(15, 15))
threshold = 0.8
sns.set_style('whitegrid', {'axes.facecolor': ".0"})
df_cluster2 = df.corr()
mask = df_cluster2.where((abs(df_cluster2) >= threshold)).isna()
plot_kws = {'s': 1}

sns.heatmap(df_cluster2,
            cmap='RdYlBu',  
            annot=True,  
            mask=mask,
            linewidth=0.20,  
            linecolor='lightgrey').set_facecolor('white')  

plt.show()  

We can see from the image that there is a strong correlation between process temperature and air temperature and a strong correlation between torque and rotational speed

*  1.4 Use module "ProfileReport" to generate detailed analysis reports of the data set

In [ ]:
from pandas_profiling import ProfileReport

profile = ProfileReport(df,
                        title="Predictive Maintenance",
                        dataset={"description": "This profiling report was generated for jiejie",
                                 "copyright_holder": "jiejie",
                                 "copyright_year": "2023"},
                        explorative=True
                        )
profile

# 2. Data Preparing

In [ ]:
df.drop(['UDI', 'Product ID'], axis=1, inplace=True)  # Delete variables that are not helpful to predict
print(df.head())

In [ ]:
df['Machine failure'].value_counts()  #

In [ ]:
df['TWF'].value_counts()

In [ ]:
df['HDF'].value_counts()

In [ ]:
df['PWF'].value_counts()

In [ ]:
df['OSF'].value_counts()

In [ ]:
df['RNF'].value_counts()

In [ ]:
# Classfication: as long as the machine works normally, it will be classified as 0, and the rest will be subdivided into five cases of machine failure
df['Machine failure'] = 0
df['Machine failure'][df['TWF'] == 1] = 1
df['Machine failure'][df['HDF'] == 1] = 2
df['Machine failure'][df['PWF'] == 1] = 3
df['Machine failure'][df['OSF'] == 1] = 4
df['Machine failure'][df['RNF'] == 1] = 5
df.drop(['TWF', 'HDF', 'PWF', 'OSF', 'RNF'], axis=1, inplace=True)
df.head()

In [ ]:
# Define some new features from the existing data
df['Power'] = df['Rotational speed [rpm]'] * df['Torque [Nm]']
df['Power wear'] = df['Power'] * df['Tool wear [min]']
df['Temperature difference'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['Temperature power'] = df['Temperature difference'] / df['Power']
df = df[['Air temperature [K]',
         'Process temperature [K]',
         'Rotational speed [rpm]',
         'Torque [Nm]',
         'Tool wear [min]',
         'Power',
         'Power wear',
         'Temperature difference',
         'Temperature power',
         'Type',
         'Machine failure'
         ]]
print(list(df))

In [ ]:
df = pd.get_dummies(df, drop_first=True)  # Achieve ont_hot_encode

In [ ]:
features = list(df.columns)  # Deal with special values and nan 

df.replace("?", np.nan, inplace=True)

df_numeric.fillna(df_numeric.mean(), inplace=True)
for feature in features:
    try:
        df[feature].filna(df[feature].mean(), inplace=True)
    except:
        try:
            df[feature].fillna(df[feature].mode(), inplace=True)
        except:
            pass
        
for feature in features:
    print(feature + "-" + str(
        len(df[df[feature].isna()])))  

# 3. Cluster analyse

In [ ]:
sns.set_style('white')
plot_kws = {"s": 1}
g = sns.pairplot(df,
                 diag_kind='hist',
                 corner=False,
                 palette='cividis',
                 hue='Machine failure')  
plt.show()  

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer
from sklearn.decomposition import PCA
pca=PCA()
X_pca=pca.fit_transform(df)  
component_names=[f"PC{i+1}" for i in range(X_pca.shape[1])]
X_pca=pd.DataFrame(X_pca,columns=component_names)
X_pca['Machine failure']=df["Machine failure"]
print(X_pca.head(10))

sns.set_style('white')
plot_kws={"s":1}
g=sns.pairplot(X_pca,diag_kind='hist',corner=False,palette='cividis',hue='Machine failure')
plt.show()  

In [ ]:
from random import sample
from numpy.random import uniform
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import scale, normalize

X = df.iloc[:, :-1]
y = df.iloc[:, -1]


# Define some new features from the original data and calculate the hopkins statistic 
# and Hopkins statistics for the dataframe X to determine the randomness of the data in space, 
# so as to determine whether the data can be clustered
def hopkins_statistic(X, k=2):
    try:
        X = X.values 
    except:
        pass
    X = normalize(X)

    sample_size = int(X.shape[0] * 0.05) 

    X_uniform_random_sample = uniform(X.min(axis=0), X.max(axis=0),
                                      (sample_size, X.shape[1]))  # A uniform random sample over the original data space

    random_indices = sample(range(0, X.shape[0], 1), k=sample_size)
    X_sample = X[random_indices]  # randomly extracted from the original data X according to random_indices

    neigh = NearestNeighbors(n_neighbors=k)  
    nbrs = neigh.fit(X)  # Initializes the unsupervised learning to implement neighbor searches

    u_distances, u_indices = nbrs.kneighbors(X_uniform_random_sample, n_neighbors=k)  
    u_distances = u_distances[:, 0]  

    w_distances, w_indices = nbrs.kneighbors(X_sample, n_neighbors=k)  
    w_distances = w_distances[:, 1]  

    u_sum = np.sum(u_distances)
    w_sum = np.sum(w_distances)

    H = u_sum / (u_sum + w_sum)
    return H


H = hopkins_statistic(X, 3)
print('Hopkins statistic:' + str(H))  

According to the scatter figure and Hopkins statistic, it is clear that use clustering analysis to explore the data set

# 4. Feature selecting

In [ ]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Obtain training data set and testing data set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

# Normalization to avoid skewed data
from sklearn.preprocessing import MinMaxScaler
sc = MinMaxScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
Selected_Features = []
import statsmodels.api as sm

def backward_regression(X, y, initial_list=[], threshold_out=0.05, verbose=True):  # threshold_out corresponds to P-values
    included = list(X.columns)
    while True:
        changed = False
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()
        # use all coefs except intercept
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max()  # null if pvalues is empty
        if worst_pval > threshold_out:
            changed = True
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            if verbose:
                print(f"worst_feature : {worst_feature}, {worst_pval} ")
        if not changed:
            break
    Selected_Features.append(included)
    print(f"\nSelected Features:\n{Selected_Features[0]}")


backward_regression(X, y)

X.head()
feature_names = list(X.columns)
print(np.shape(X), len(feature_names)) 

# 5. Data modeling

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_auc_score, \
    plot_confusion_matrix, matthews_corrcoef
import time

model_performance = pd.DataFrame(columns=['Accuracy', 'Recall', 'Precision', 'F1-Score', 'MCC score',
                                          'time to train', 'time to predict', 'total time'])

In [ ]:
# KNN model
from sklearn.neighbors import KNeighborsClassifier

start = time.time()
model = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
end_train = time.time()
y_pred = model.predict(X_test)
end_predict = time.time()

accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
f1s = f1_score(y_test, y_pred)
MCC = matthews_corrcoef(y_test, y_pred)

print("Accuracy: " + "{:.2%}".format(accuracy),
      "\nRecall: " + "{:.2%}".format(recall),
      "\nPrecison: " + "{:.2%}".format(precision),
      "\nF1-Score: " + "{:.2%}".format(f1s),
      "\nMCC: " + "{:.2%}".format(MCC),
      "\ntime to train: " + "{:.2f}".format(end_train - start) + " s",
      "\ntime to predict: " + "{:.2f}".format(end_predict - end_train) + " s",
      "\ntotal: " + "{:.2f}".format(end_predict - start) + " s"
      )
model_performance.loc['Logistic'] = [accuracy, recall, precision, f1s, MCC, end_train - start, end_predict - end_train,
                                     end_predict - start]

plt.rcParams['figure.figsize'] = 5, 5
plt.rcParams['font.size'] = 10
sns.set_style('white')
plot_confusion_matrix(model, X_test, y_test, cmap=plt.cm.Blues)
plt.show()

In [ ]:
# Decidion Tree
from sklearn.tree import DecisionTreeClassifier

start = time.time()
model = DecisionTreeClassifier().fit(X_train, y_train)
end_train = time.time()
y_pred = model.predict(X_test)
end_predict = time.time()

accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
f1s = f1_score(y_test, y_pred)
MCC = matthews_corrcoef(y_test, y_pred)

print("Accuracy: " + "{:.2%}".format(accuracy),
      "\nRecall: " + "{:.2%}".format(recall),
      "\nPrecison: " + "{:.2%}".format(precision),
      "\nF1-Score: " + "{:.2%}".format(f1s),
      "\nMCC: " + "{:.2%}".format(MCC),
      "\ntime to train: " + "{:.2f}".format(end_train - start) + " s",
      "\ntime to predict: " + "{:.2f}".format(end_predict - end_train) + " s",
      "\ntotal: " + "{:.2f}".format(end_predict - start) + " s"
      )
model_performance.loc['Decision Tree'] = [accuracy, recall, precision, f1s, MCC, end_train - start,
                                          end_predict - end_train, end_predict - start]

plt.rcParams['figure.figsize'] = 5, 5
plt.rcParams['font.size'] = 10
sns.set_style('white')
plot_confusion_matrix(model, X_test, y_test, cmap=plt.cm.Blues)
plt.show()

plt.rcParams['figure.figsize'] = 10, 10
sns.set_style('white')
feat_importances = pd.Series(model.feature_importances_, index=feature_names)
feat_importances = feat_importances.groupby(level=0).mean() 
feat_importances.nlargest(20).plot(kind='barh').invert_yaxis()  
sns.despine()  
plt.show()

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

start = time.time()
model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0, bootstrap=True).fit(X_train, y_train)
end_train = time.time()
y_pred = model.predict(X_test)
end_predict = time.time()

accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
f1s = f1_score(y_test, y_pred)
MCC = matthews_corrcoef(y_test, y_pred)

print("Accuracy: " + "{:.2%}".format(accuracy),
      "\nRecall: " + "{:.2%}".format(recall),
      "\nPrecison: " + "{:.2%}".format(precision),
      "\nF1-Score: " + "{:.2%}".format(f1s),
      "\nMCC: " + "{:.2%}".format(MCC),
      "\ntime to train: " + "{:.2f}".format(end_train - start) + " s",
      "\ntime to predict: " + "{:.2f}".format(end_predict - end_train) + " s",
      "\ntotal: " + "{:.2f}".format(end_predict - start) + " s"
      )
model_performance.loc['Random Forest'] = [accuracy, recall, precision, f1s, MCC, end_train - start,
                                          end_predict - end_train, end_predict - start]

plt.rcParams['figure.figsize'] = 5, 5
plt.rcParams['font.size'] = 10
sns.set_style('white')
plot_confusion_matrix(model, X_test, y_test, cmap=plt.cm.Blues)
plt.show()

In [ ]:
# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

start = time.time()
model = GradientBoostingClassifier().fit(X_train, y_train)
end_train = time.time()
y_pred = model.predict(X_test)
end_predict = time.time()

accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
f1s = f1_score(y_test, y_pred)
MCC = matthews_corrcoef(y_test, y_pred)

print("Accuracy: " + "{:.2%}".format(accuracy),
      "\nRecall: " + "{:.2%}".format(recall),
      "\nPrecison: " + "{:.2%}".format(precision),
      "\nF1-Score: " + "{:.2%}".format(f1s),
      "\nMCC: " + "{:.2%}".format(MCC),
      "\ntime to train: " + "{:.2f}".format(end_train - start) + " s",
      "\ntime to predict: " + "{:.2f}".format(end_predict - end_train) + " s",
      "\ntotal: " + "{:.2f}".format(end_predict - start) + " s"
      )
model_performance.loc['Gradient Boosting'] = [accuracy, recall, precision, f1s, MCC, end_train - start,
                                              end_predict - end_train, end_predict - start]

plt.rcParams['figure.figsize'] = 5, 5
plt.rcParams['font.size'] = 10
sns.set_style('white')
plot_confusion_matrix(model, X_test, y_test, cmap=plt.cm.Blues)
plt.show()

plt.rcParams['figure.figsize'] = 10, 10
sns.set_style('white')
feat_importances = pd.Series(model.feature_importances_, index=feature_names)
feat_importances = feat_importances.groupby(level=0).mean()
feat_importances.nlargest(20).plot(kind='barh').invert_yaxis()
sns.despine()
plt.show()

In [ ]:
# Neural Network MLP
from sklearn.neural_network import MLPClassifier

start = time.time()
model = MLPClassifier(hidden_layer_sizes=(25, 25, 25),
                      activation='relu',
                      solver='adam',
                      batch_size=2000,
                      verbose=0).fit(X_train, y_train)
end_train = time.time()
y_pred = model.predict(X_test)
end_predict = time.time()

accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
f1s = f1_score(y_test, y_pred)
MCC = matthews_corrcoef(y_test, y_pred)

print("Accuracy: " + "{:.2%}".format(accuracy),
      "\nRecall: " + "{:.2%}".format(recall),
      "\nPrecison: " + "{:.2%}".format(precision),
      "\nF1-Score: " + "{:.2%}".format(f1s),
      "\nMCC: " + "{:.2%}".format(MCC),
      "\ntime to train: " + "{:.2f}".format(end_train - start) + " s",
      "\ntime to predict: " + "{:.2f}".format(end_predict - end_train) + " s",
      "\ntotal: " + "{:.2f}".format(end_predict - start) + " s"
      )
model_performance.loc['Neural Network MLP'] = [accuracy, recall, precision, f1s, MCC, end_train - start,
                                               end_predict - end_train, end_predict - start]

plt.rcParams['figure.figsize'] = 5, 5
plt.rcParams['font.size'] = 10
sns.set_style('white')
plot_confusion_matrix(model, X_test, y_test, cmap=plt.cm.Blues)
plt.show()

In [ ]:
model_performance.fillna(.90, inplace=True)
model_performance.style.background_gradient(cmap='coolwarm').format({'Accuracy': '{:.2%}',
                                                                     'Precision': '{:.2%}',
                                                                     'Recall': '{:.2%}',
                                                                     'F1-Score': '{:.2%}',
                                                                     'MCC score': '{:.2%}',
                                                                     'time to train': '{:.1f}',
                                                                     'time to predict': '{:.1f}',
                                                                     'total time': '{:.1f}',
                                                                     })
print(model_performance)

From above model, we can learn that Gradient Boosting classfieir and Neural Network MLP seem to be the go-to models for this case, giving a decent MCC score.